# 01 · Bronze Ingest

Read all raw CSVs from `01_data/raw/` (uploaded to the lakehouse `Files/raw/`) into
Delta tables in the **bronze** lakehouse, preserving the source schema.

Per the architecture, bronze keeps the original records intact and adds lineage
metadata (`_source_file`, `_ingest_timestamp`) partitioned by `ingest_date`.

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Raw CSVs are uploaded to the lakehouse Files area under raw/
RAW_DIR: str = "Files/raw"
BRONZE_PREFIX: str = "bronze_"

# One bronze Delta table per source file (source schema preserved as-is)
SOURCES: list[str] = [
    "farmers",
    "lots",
    "lot_consumption",
    "production_runs",
    "quality_tests",
    "sales_orders",
]


def ingest_csv_to_bronze(name: str) -> int:
    """Load a raw CSV into a bronze Delta table, preserving the source schema.

    Returns the number of rows written.
    """
    src_path = f"{RAW_DIR}/{name}.csv"
    table_name = f"{BRONZE_PREFIX}{name}"

    # Infer the source schema; multiLine + quote/escape keep Vietnamese text intact
    df: DataFrame = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .csv(src_path)
    )

    # Add bronze lineage metadata without altering source columns
    df = (
        df.withColumn("_source_file", F.input_file_name())
          .withColumn("_ingest_timestamp", F.current_timestamp())
          .withColumn("ingest_date", F.current_date())
    )

    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .partitionBy("ingest_date")
          .saveAsTable(table_name)
    )

    row_count = df.count()
    print(f"Wrote {row_count:,} rows -> {table_name}")
    return row_count


total = sum(ingest_csv_to_bronze(name) for name in SOURCES)
print(f"\nBronze ingest complete: {len(SOURCES)} tables, {total:,} total rows.")